<a href="https://colab.research.google.com/github/unknxwNS/Ihopeitworks/blob/main/%D0%9C%D1%96%D1%85%D0%B0%D0%BB%D1%8C%D0%BE%D0%B2_%D0%9B%D0%A0_9_%D0%A0%D0%B5%D0%BA%D0%BE%D0%BC%D0%B5%D0%BD%D0%B4%D0%B0%D1%86%D1%96%D0%B9%D0%BD%D1%96_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B8_%D0%A8%D0%B0%D0%B1%D0%BB%D0%BE%D0%BD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Міхальов Андрій Володимирович

**Група:** ФІТ 4-10

**Дата:** 19.11.26

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [4]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:
# Кількість унікальних користувачів
print("Кількість користувачів:", ratings_df['user_id'].nunique())

# Кількість унікальних фільмів
print("Кількість фільмів:", ratings_df['item_id'].nunique())

# Статистика оцінок
print("Мінімальна оцінка:", ratings_df['rating'].min())
print("Максимальна оцінка:", ratings_df['rating'].max())
print("Середня оцінка:", ratings_df['rating'].mean())

# Перевірка пропущених значень
print("\nПропущені значення:")
print(ratings_df.isnull().sum())

Кількість користувачів: 943
Кількість фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.52986

Пропущені значення:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64


## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [5]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

print("Розмір навчальної вибірки:", train_df.shape)
print("Розмір тестової вибірки:", test_df.shape)

Розмір навчальної вибірки: (80000, 4)
Розмір тестової вибірки: (20000, 4)


In [6]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:
# 1. Створення Reader із діапазоном оцінок
reader = Reader(rating_scale=data.reader.rating_scale)

# 2. Створення навчального набору для Surprise
train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)

# 3. Формування тестового набору у вигляді списку кортежів
testset = list(
    test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None)
)

# 4. Перевірка розмірів
print("Розмір train:", len(train_df))
print("Розмір test:", len(testset))

Розмір train: 80000
Розмір test: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [7]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [8]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:
# Параметри для підбору SVD
param_grid_svd = {
    'n_factors': [50, 100, 150],
    'n_epochs': [20, 30],
    'lr_all': [0.005, 0.01],
    'reg_all': [0.02, 0.1]
}

# Підбір гіперпараметрів
gs_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

# Навчання та пошук найкращих параметрів
gs_svd.fit(train_data)

print("Найкращі параметри за RMSE:", gs_svd.best_params['rmse'])
print("Найкращий RMSE:", gs_svd.best_score['rmse'])
print("Найкращий MAE:", gs_svd.best_score['mae'])

Найкращі параметри за RMSE: {'n_factors': 150, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}
Найкращий RMSE: 0.9314642476997923
Найкращий MAE: 0.73666077197399


In [9]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:
# Параметри для підбору KNNBasic
param_grid_knn = {
    'k': [20, 40, 60],
    'min_k': [1, 3, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [True, False]
    }
}

# Підбір гіперп параметрів
gs_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

# Навчання та пошук найкращих параметрів
gs_knn.fit(train_data)

print("Найкращі параметри за RMSE:", gs_knn.best_params['rmse'])
print("Найкращий RMSE:", gs_knn.best_score['rmse'])
print("Найкращий MAE:", gs_knn.best_score['mae'])

Найкращі параметри за RMSE: {'k': 60, 'min_k': 3, 'sim_options': {'name': 'cosine', 'user_based': True}}
Найкращий RMSE: 1.0251419299452005
Найкращий MAE: 0.8140065128674161


In [10]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:
# Результати SVD
print("===== SVD =====")
print("Найкращий RMSE:", gs_svd.best_score['rmse'])
print("Параметри RMSE:", gs_svd.best_params['rmse'])
print("Найкращий MAE:", gs_svd.best_score['mae'])
print("Параметри MAE:", gs_svd.best_params['mae'])

# Результати KNNBasic
print("\n===== KNNBasic =====")
print("Найкращий RMSE:", gs_knn.best_score['rmse'])
print("Параметри RMSE:", gs_knn.best_params['rmse'])
print("Найкращий MAE:", gs_knn.best_score['mae'])
print("Параметри MAE:", gs_knn.best_params['mae'])

===== SVD =====
Найкращий RMSE: 0.9314642476997923
Параметри RMSE: {'n_factors': 150, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}
Найкращий MAE: 0.73666077197399
Параметри MAE: {'n_factors': 150, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}

===== KNNBasic =====
Найкращий RMSE: 1.0251419299452005
Параметри RMSE: {'k': 60, 'min_k': 3, 'sim_options': {'name': 'cosine', 'user_based': True}}
Найкращий MAE: 0.8140065128674161
Параметри MAE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | ... | ... | ... |
| KNNBasic | ... | ... | ... |

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [11]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:
# TODO 6.1 — Найкращі параметри за CV RMSE
best_svd_params = gs_svd.best_params['rmse']
best_knn_params = gs_knn.best_params['rmse']

print("SVD parameters:", best_svd_params)
print("KNN parameters:", best_knn_params)

# TODO 6.2 — Створення trainset
trainset = train_data.build_full_trainset()

# TODO 6.3 — Навчання моделей із найкращими параметрами
best_svd = SVD(**best_svd_params, random_state=42)
best_knn = KNNBasic(**best_knn_params)

best_svd.fit(trainset)
best_knn.fit(trainset)

print("\nМоделі успішно навчені.")

SVD parameters: {'n_factors': 150, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}
KNN parameters: {'k': 60, 'min_k': 3, 'sim_options': {'name': 'cosine', 'user_based': True}}
Computing the cosine similarity matrix...
Done computing similarity matrix.

Моделі успішно навчені.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [12]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:
# TODO 7.1 — Прогнозування на тестовій вибірці
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

# TODO 7.2 — Оцінювання моделей
svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

# Виведення результатів
print("===== SVD =====")
print(f"RMSE: {svd_rmse:.4f}")
print(f"MAE:  {svd_mae:.4f}")

print("\n===== KNNBasic =====")
print(f"RMSE: {knn_rmse:.4f}")
print(f"MAE:  {knn_mae:.4f}")

===== SVD =====
RMSE: 0.9116
MAE:  0.7201

===== KNNBasic =====
RMSE: 1.0096
MAE:  0.7992


In [13]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
})

results = results.sort_values(by='RMSE').reset_index(drop=True)

display(results)

,Model,RMSE,MAE
0,SVD,0.911621,0.720121
1,KNNBasic,1.009639,0.799197


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | ... | ... |
| KNNBasic | ... | ... |

**Порівняння з CV:** _Прокоментуйте, чи суттєво відрізняються оцінки на кросвалідації та незалежній тестовій вибірці._

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [14]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:
# Початковий ID користувача
user_id = '196'

# TODO 8.1 — Визначення найкращої моделі за CV RMSE
best_model_name = min(
    ['SVD', 'KNNBasic'],
    key=lambda name: (
        gs_svd.best_score['rmse']
        if name == 'SVD'
        else gs_knn.best_score['rmse']
    )
)

best_model = best_svd if best_model_name == 'SVD' else best_knn
print("Найкраща модель:", best_model_name)

# TODO 8.2 — Перевірка наявності користувача у trainset
try:
    inner_uid = best_model.trainset.to_inner_uid(user_id)

    # TODO 8.3 — Пошук фільмів, які користувач ще не оцінював
    rated_items = set(
        iid for (iid, rating) in best_model.trainset.ur[inner_uid]
    )

    all_items = set(best_model.trainset.all_items())
    unrated_items = all_items - rated_items

    # TODO 8.4 — Прогноз оцінок для неоцінених фільмів
    predictions = [
        (item_id, best_model.predict(user_id, item_id).est)
        for item_id in unrated_items
    ]

    recommendations = pd.DataFrame(
        predictions,
        columns=['item_id', 'predicted_rating']
    )

    recommendations = recommendations.sort_values(
        by='predicted_rating',
        ascending=False
    ).head(10).reset_index(drop=True)

    display(recommendations)

except ValueError:
    print(f"Користувач {user_id} відсутній у навчальній вибірці.")

Найкраща модель: SVD


,item_id,predicted_rating
0,0,3.550585
1,1,3.550585
2,2,3.550585
3,4,3.550585
4,5,3.550585
5,6,3.550585
6,7,3.550585
7,9,3.550585
8,10,3.550585
9,11,3.550585


## 9. Перевірка виконання

Позначте після реалізації:

- [ ] Дані `ml-100k` завантажено та перевірено.
- [ ] Є розбиття 80/20 з фіксованим `random_state`.
- [ ] `Reader`, `train_data` і `testset` створені коректно.
- [ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ ] Для SVD і KNNBasic показано параметри та результати CV.
- [ ] Моделі навчено на повній навчальній частині.
- [ ] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ] Наведено порівняльну таблицю та інтерпретацію.
- [ ] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

У ході лабораторної роботи було реалізовано та порівняно дві моделі рекомендацій — SVD та KNNBasic. За результатами крос-валідації кращою виявилася модель SVD, оскільки її RMSE становив 0.9315, а MAE — 0.7367, що менше, ніж у KNNBasic. Для SVD найкращими параметрами стали 150 факторів, 30 епох, швидкість навчання 0.01 та регуляризація 0.1. Для KNNBasic оптимальними параметрами були 40 сусідів, мінімум 3 сусіди, косинусна схожість та підхід на основі користувачів. На тестовій вибірці SVD також показала кращий результат з RMSE 0.9116 та MAE 0.7201, тоді як KNNBasic отримала RMSE 1.0096 та MAE 0.7992. Результати на тестовій вибірці близькі до показників крос-валідації, що свідчить про стабільність моделей. Отже, для формування рекомендацій було обрано SVD, оскільки вона точніше прогнозує оцінки користувачів. Основним обмеженням системи є проблема нових користувачів і фільмів, для яких відсутня історія оцінок.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**